# InternScenes -> USD scene converter

Pick a **scene number**, then run the notebook to build a scene in the exact
same format as the reference `issacsim-assets/scene0000_00`:

```
issacsim-assets/<SCENE_ID>/glb_scene.glb
issacsim-assets/<SCENE_ID>/usd/scene.usd
issacsim-assets/<SCENE_ID>/usd/textures/*.jpg|png
```

Pipeline (each stage is its own cell, or run the single **RUN** cell at the end):

1. ensure `layout.json` (download/extract from the HF `Layout_info.tar.gz` if missing)
2. download per-object assets (`--no-archives` by default; opt-in `partnet_mobility`/`objaverse`)
3. stage `StructureMesh` (floor/wall/ceiling) into the compose output dir
4. **compose** the scene GLB (`SceneComposer`)
5. **GLB -> USD** via Isaac Sim (`glb2usd_headless.py`, headless, foreground)
6. copy the GLB to `issacsim-assets/<SCENE_ID>/` and report artifacts

> NOTE: `scene0000_03 ... scene0000_09` do **not** exist in the HF dataset.
> The dataset's scannet scenes are `scene0000_00/01/02`, `scene0001_00/01`,
> `scene0002_00/01`, `scene0003_00/01/02`, `scene0004_00`, `scene0005_00`, ...
> Pick one of *those* (or any id whose `layout.json` is present).

## How to use
1. Run the **Configuration** cell.
2. Run the **Select scene** cell and pick a scene in the dropdown (or edit `SCENE_ID`).
3. Run the **RUN (full pipeline)** cell.
   - It is safe to re-run: every stage checks what already exists.
4. The result lands in `issacsim-assets/<SCENE_ID>/`.

Individual stage cells (below) let you run/debug one step at a time.

## Configuration

In [ ]:
# ---------- paths (edit only if your layout differs) ----------
import os, sys, time, json, shutil, subprocess, tarfile
from pathlib import Path

PROJECT     = Path("/home/snt/projects/AgenticMemoryNav")
INTERN      = PROJECT / "external-lib" / "InternScenes"
COMPOSE_DIR = INTERN / "InternScenes" / "InternScenes_Real2Sim"  # dir containing compose_scenes.py
GLB2USD_PY  = INTERN / "glb2usd_headless.py"
PYTHON      = INTERN / ".venv" / "bin" / "python"   # has trimesh + huggingface_hub
ISAAC_PY    = Path("/home/snt/isaacsim/python.sh")  # Isaac Sim, for GLB -> USD
ASSETS      = PROJECT / "issacsim-assets"           # final destination
DATA        = INTERN / "data"
LAYOUT_DIR  = DATA / "Layout_info"
COMPOSED    = INTERN / "tutorial" / "examples" / "composed_scenes"
BASE_DIR    = COMPOSE_DIR.parent                   # compose_scenes.py sets BASE_DIR = Path(cwd).parent

ok = all(p.exists() for p in [PYTHON, GLB2USD_PY, COMPOSE_DIR, ASSETS])
print("paths OK" if ok else "MISSING PATHS")

# ---------- options (edit these) ----------
DATASET        = "scannet"   # layout sub-folder: scannet / 3rscan / ...
WITH_PARTNET   = True        # fetch 0.72 GB partnet_mobility archive (recommended; shared)
WITH_OBJAVERSE = False       # fetch 101 GB objaverse archive (very large, archive-only)
SCENE_ID       = "scene0000_01"   # <-- or pick from the dropdown in the next cell

## Helpers

In [ ]:
# ---------- helpers (use the globals SCENE_ID, DATASET, WITH_PARTNET, WITH_OBJAVERSE) ----------
def ensure_symlinks():
    # compose_scenes.py expects BASE_DIR/data and BASE_DIR/tutorial; in this checkout
    # data/ and tutorial/ live at external-lib/InternScenes/, so symlink them into BASE_DIR.
    for name in ("data", "tutorial"):
        link = BASE_DIR / name
        target = INTERN / name
        if link.exists() or link.is_symlink():
            print("symlink exists: %s -> %s" % (link, link.resolve()))
        else:
            link.symlink_to(target, target_is_directory=True)
            print("created symlink %s -> %s" % (link, target))

COMPOSE_RUNNER = COMPOSE_DIR / "_nb_compose.py"
def write_compose_runner():
    code = (
        "import os, sys\n"
        "sys.path.insert(0, os.getcwd())\n"
        "import compose_scenes\n"
        "scene = sys.argv[1]\n"
        "sc = compose_scenes.SceneComposer()\n"
        "sc.compose_one_scene(scene, use_texture=True, add_floor=True, add_wall=True, add_ceiling=True)\n"
        "print('COMPOSE_DONE', scene)\n"
    )
    COMPOSE_RUNNER.write_text(code)
    print("wrote", COMPOSE_RUNNER)

ensure_symlinks()
write_compose_runner()
print("helpers ready")

## Select scene

In [ ]:
# ---------- select the scene ----------
have_layout = sorted(p.parent.name for p in (LAYOUT_DIR / DATASET).glob("*/layout.json"))
print("available layouts in %s/ : %s" % (DATASET, have_layout))

try:
    import ipywidgets as widgets
    from IPython.display import display
    options = have_layout if have_layout else [SCENE_ID]
    SCENE_W = widgets.Dropdown(options=options, value=options[0], description="scene")
    PM_W    = widgets.Checkbox(value=WITH_PARTNET,  description="fetch 0.72 GB partnet_mobility")
    OBJ_W   = widgets.Checkbox(value=WITH_OBJAVERSE, description="fetch 101 GB objaverse (archive only)")
    display(SCENE_W, PM_W, OBJ_W)
    HAVE_WIDGETS = True
    print("pick a scene in the dropdown, then run the RUN cell.")
except Exception:
    HAVE_WIDGETS = False
    print("no ipywidgets -> edit SCENE_ID in the Configuration cell.")

def current_selection():
    # returns (SCENE_ID, WITH_PARTNET, WITH_OBJAVERSE) from widget or plain vars
    if HAVE_WIDGETS:
        return SCENE_W.value, PM_W.value, OBJ_W.value
    return SCENE_ID, WITH_PARTNET, WITH_OBJAVERSE

## Stage 1 - layout

In [ ]:
# ---------- Stage 1: ensure layout.json ----------
def stage_1_layout():
    global SCENE_ID, DATASET, WITH_PARTNET, WITH_OBJAVERSE
    SCENE_ID, WITH_PARTNET, WITH_OBJAVERSE = current_selection()
    print("SCENE_ID=%s DATASET=%s WITH_PARTNET=%s WITH_OBJAVERSE=%s"
          % (SCENE_ID, DATASET, WITH_PARTNET, WITH_OBJAVERSE))
    layout_path = LAYOUT_DIR / DATASET / SCENE_ID / "layout.json"
    if layout_path.exists():
        print("layout present: %s (%d instances)" % (layout_path, len(json.load(open(layout_path)))))
        return layout_path
    print("%s layout.json missing -> extracting from HF Layout_info.tar.gz" % SCENE_ID)
    lay_archive = DATA / ".hf_layout" / "Layout_info.tar.gz"
    if not lay_archive.exists():
        from huggingface_hub import hf_hub_download
        print("downloading 2.96 GB Layout_info.tar.gz (one time)...")
        lay_archive = hf_hub_download("InternRobotics/InternScenes", "Layout_info.tar.gz",
                                      repo_type="dataset", local_dir=str(DATA / ".hf_layout"))
    tf = tarfile.open(lay_archive, "r:gz")
    n = 0
    for m in tf.getmembers():
        if m.name == ("Layout_info/%s/%s/layout.json" % (DATASET, SCENE_ID)) \
           or m.name.startswith("Layout_info/%s/%s/" % (DATASET, SCENE_ID)):
            tf.extract(m, str(DATA)); n += 1
    tf.close()
    layout_path = LAYOUT_DIR / DATASET / SCENE_ID / "layout.json"
    if not layout_path.exists():
        avail = sorted(p.parent.name for p in (LAYOUT_DIR / DATASET).glob("*/layout.json"))
        print("CONCLUSION: %s does NOT exist in the dataset. Available: %s" % (SCENE_ID, avail))
        raise SystemExit("aborting: layout.json not found for %s" % SCENE_ID)
    print("extracted %d members; layout now present: %s (%d instances)"
          % (n, layout_path, len(json.load(open(layout_path)))))
    return layout_path

stage_1_layout()

## Stage 2 - assets

In [ ]:
# ---------- Stage 2: download per-object assets ----------
def stage_2_assets(layout_path):
    global WITH_PARTNET, WITH_OBJAVERSE
    objs = json.load(open(layout_path))
    uids = sorted({e["model_uid"] for e in objs if "model_uid" in e})
    per_object, archives, skipped = [], [], {}
    for uid in uids:
        lib = uid.split("/")[0]
        if lib == "partnet_mobility":
            if WITH_PARTNET:
                archives.append("asset_library/partnet_mobility/partnet_mobility.tar.gz")
            else:
                skipped["partnet_mobility"] = skipped.get("partnet_mobility", 0) + 1
            continue
        if lib == "objaverse":
            if WITH_OBJAVERSE:
                for i in range(10):
                    archives.append("asset_library/objaverse/objaverse.tar.gz.%02d" % i)
            else:
                skipped["objaverse"] = skipped.get("objaverse", 0) + 1
            continue
        per_object.append("asset_library/%s.glb" % uid)
    patterns = per_object + list(dict.fromkeys(archives))
    print("instances=%d unique_assets=%d per_object=%d archives=%d skipped=%s"
          % (len(objs), len(uids), len(per_object), len(set(archives)), skipped))
    to_get = [p for p in patterns if not (DATA / p).exists()]
    print("need to download: %d (already present: %d)" % (len(to_get), len(patterns) - len(to_get)))
    if to_get:
        from huggingface_hub import snapshot_download
        t0 = time.time()
        snapshot_download("InternRobotics/InternScenes", repo_type="dataset",
                          allow_patterns=to_get, local_dir=str(DATA))
        print("downloaded %d files in %.0fs" % (len(to_get), time.time() - t0))
    # partnet_mobility archive -> data/asset_library/partnet_mobility/ (compose wants it there)
    pm_dir = DATA / "asset_library" / "partnet_mobility"
    want_partnet = any("partnet_mobility" in a for a in archives)
    have_partnet = pm_dir.exists() and any(pm_dir.glob("*/whole.glb"))
    if want_partnet and not have_partnet:
        pm = DATA / ".hf_pm" / "asset_library" / "partnet_mobility" / "partnet_mobility.tar.gz"
        if not pm.exists():
            from huggingface_hub import hf_hub_download
            print("downloading 0.72 GB partnet_mobility archive...")
            pm = hf_hub_download("InternRobotics/InternScenes",
                                "asset_library/partnet_mobility/partnet_mobility.tar.gz",
                                repo_type="dataset", local_dir=str(DATA / ".hf_pm"))
        print("extracting partnet_mobility archive (0.72 GB)...")
        with tarfile.open(pm, "r:gz") as tf:
            tf.extractall(str(DATA))           # -> data/partnet_mobility/...
        extracted = DATA / "partnet_mobility"
        if extracted.exists() and not pm_dir.exists():
            pm_dir.parent.mkdir(parents=True, exist_ok=True)
            extracted.rename(pm_dir)
        print("partnet_mobility at %s (%d whole.glb)" % (pm_dir, len(list(pm_dir.glob("*/whole.glb")))))
    print("Stage 2 done")

stage_2_assets(layout_path if 'layout_path' in dir() else LAYOUT_DIR / DATASET / SCENE_ID / "layout.json")

## Stage 3 - structure mesh

In [ ]:
# ---------- Stage 3: stage StructureMesh (floor/wall/ceiling) ----------
def stage_3_structure():
    src = LAYOUT_DIR / DATASET / SCENE_ID / "StructureMesh"
    dst = COMPOSED / DATASET / SCENE_ID / "StructureMesh"
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print("staged StructureMesh -> %s (%d glbs)" % (dst, len(list(dst.glob('*.glb')))))
    elif dst.exists():
        print("StructureMesh already staged at %s" % dst)
    else:
        print("WARN: no StructureMesh for %s (floor/wall/ceiling will be skipped)" % SCENE_ID)

stage_3_structure()

## Stage 4 - compose GLB

In [ ]:
# ---------- Stage 4: compose the scene GLB ----------
def stage_4_compose():
    out_glb = COMPOSED / DATASET / SCENE_ID / "glb_scene.glb"
    scene_arg = "%s/%s" % (DATASET, SCENE_ID)
    t0 = time.time()
    r = subprocess.run([str(PYTHON), str(COMPOSE_RUNNER), scene_arg],
                       cwd=str(COMPOSE_DIR), capture_output=True, text=True)
    print(r.stdout[-1500:])
    if r.stderr:
        print("STDERR tail:", r.stderr[-800:])
    print("compose rc=%d elapsed=%.0fs" % (r.returncode, time.time() - t0))
    if r.returncode != 0:
        raise RuntimeError("compose failed (rc=%d)" % r.returncode)
    if not out_glb.exists():
        raise RuntimeError("no glb produced")
    print("composed glb: %s (%.2f GB)" % (out_glb, out_glb.stat().st_size / 1e9))
    return out_glb

out_glb = stage_4_compose()

## Stage 5 - GLB to USD

In [ ]:
# ---------- Stage 5: GLB -> USD via Isaac Sim (headless, foreground) ----------
def stage_5_usd(out_glb):
    usd_dir = ASSETS / SCENE_ID / "usd"
    usd_dir.mkdir(parents=True, exist_ok=True)
    out_usd = usd_dir / "scene.usd"
    t0 = time.time()
    print("GLB -> USD via Isaac Sim (headless; may take a few minutes)...")
    r = subprocess.run([str(ISAAC_PY), str(GLB2USD_PY), "--file", str(out_glb), "--out", str(out_usd)],
                       capture_output=True, text=True)
    print((r.stdout + "\n" + r.stderr)[-2500:])
    print("usd rc=%d elapsed=%.0fs" % (r.returncode, time.time() - t0))
    if r.returncode != 0 or not out_usd.exists():
        raise RuntimeError("GLB->USD failed (rc=%d)" % r.returncode)
    print("usd: %s (%.2f GB)" % (out_usd, out_usd.stat().st_size / 1e9))
    return out_usd

out_usd = stage_5_usd(out_glb)

## Stage 6 - finalize + report

In [ ]:
# ---------- Stage 6: copy GLB to issacsim-assets + report ----------
def stage_6_finalize(out_glb, out_usd):
    dst_glb = ASSETS / SCENE_ID / "glb_scene.glb"
    dst_glb.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(out_glb, dst_glb)
    tex = (ASSETS / SCENE_ID / "usd" / "textures")
    ntex = len(list(tex.glob("*"))) if tex.exists() else 0
    def _mb(p):
        return "%.1f MB" % (p.stat().st_size / 1e6) if p.exists() else "MISSING"
    print("=== %s -> %s ===" % (SCENE_ID, ASSETS / SCENE_ID))
    print("  glb_scene.glb : %s" % _mb(dst_glb))
    print("  usd/scene.usd : %s" % _mb(out_usd))
    print("  usd/textures/ : %d files" % ntex)
    print("  matches reference format: %s" % (dst_glb.exists() and out_usd.exists() and ntex > 0))

stage_6_finalize(out_glb, out_usd)

## RUN (full pipeline)

In [ ]:
# ---------- RUN (full pipeline in one cell) ----------
# re-read the current selection (widget or plain vars)
SCENE_ID, WITH_PARTNET, WITH_OBJAVERSE = current_selection()
print(">>> %s / %s  (partnet=%s, objaverse=%s)"
      % (DATASET, SCENE_ID, WITH_PARTNET, WITH_OBJAVERSE))

layout_path = stage_1_layout()
stage_2_assets(layout_path)
stage_3_structure()
out_glb = stage_4_compose()
out_usd = stage_5_usd(out_glb)
stage_6_finalize(out_glb, out_usd)
print("DONE -> %s" % (ASSETS / SCENE_ID))

## Notes / troubleshooting
- **`layout.json not found`**: the scene id is not in the dataset. Use the
  dropdown, or check the list printed by the *Select scene* cell.
  (`scene0000_03 ... _09` are NOT in the dataset.)
- **compose errors like `string is not a file: .../objaverse/....glb`**:
  expected when `WITH_OBJAVERSE=False` (the 101 GB objaverse archive is not
  fetched). trimesh skips those objects; the scene is otherwise complete.
- **partnet_mobility**: fetched by default (`WITH_PARTNET=True`, 0.72 GB, shared)
  and extracted into `data/asset_library/partnet_mobility/`.
- **GLB->USD** runs Isaac Sim headless and **must be foreground** (the box reaper
  SIGKILLs background Isaac Sim processes). This notebook cell is foreground, so
  it is fine. Do not run it as a background process.
- **Disk**: the layout archive (2.96 GB) and partnet archive (0.72 GB) are cached
  under `data/.hf_layout` and `data/.hf_pm`; delete them to reclaim ~3.7 GB.
- **Reference**: `issacsim-assets/scene0000_00` (glb 339 MB, usd 424 MB,
  132 textures) is the golden sample.